In [28]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con) 

In [29]:
q("""SELECT name FROM sqlite_master                 
WHERE  type = 'table'                  
ORDER  BY name; """)

,name
0,circles
1,complaints
2,customers
3,plans
4,recharges
5,usage_daily


In [30]:
q("""SELECT 'circles' AS tables, COUNT(*) AS n FROM circles   
UNION ALL SELECT 'plans',     COUNT(*) FROM plans    
UNION ALL SELECT 'customers',   COUNT(*) FROM customers 
UNION ALL SELECT 'usage_daily', COUNT(*) FROM usage_daily
UNION ALL SELECT 'recharges',   COUNT(*) FROM recharges
UNION ALL SELECT 'complaints',  COUNT(*) FROM complaints; """)

,tables,n
0,circles,22
1,plans,14
2,customers,22000
3,usage_daily,1326739
4,recharges,408089
5,complaints,26841


In [31]:
import sqlite3                                 
import pandas as pd                             
pcon = sqlite3.connect('../data/practice/practice.db') 
pd.read_sql('SELECT * FROM loans', pcon) 
def pq(sql):                                     
    return pd.read_sql(sql, pcon)     

In [32]:
pq("""SELECT member_id,                                 -- one row each
       COUNT(*)                 AS loans,         -- all loans
       SUM(returned_on IS NULL) AS still_out      -- not back yet
FROM   loans GROUP BY member_id;                  -- five rows
""")

,member_id,loans,still_out
0,1,3,1
1,2,3,1
2,3,2,1
3,4,1,0
4,5,1,0


In [33]:
q("""SELECT name FROM sqlite_master                 
WHERE  type = 'table'                  
ORDER  BY name; """)

,name
0,circles
1,complaints
2,customers
3,plans
4,recharges
5,usage_daily


In [34]:
pq("""SELECT * FROM shop_customers;""")

,customer_id,name,city,joined_on
0,1,Asha,Kochi,2024-01-15
1,2,Ravi,Chennai,2024-03-02
2,3,Meera,Kochi,2024-06-20
3,4,Arjun,Pune,2024-09-10
4,5,Fatima,Chennai,2025-01-05
5,6,John,Pune,2025-02-14


In [35]:
pq("""SELECT * FROM shop_orders;  """)

,order_id,customer_id,order_date,amount
0,101,1,2025-01-03,450
1,102,1,2025-01-20,300
2,103,2,2025-01-11,1200
3,104,3,2025-02-02,250
4,105,1,2025-02-15,600
5,106,4,2025-02-21,900
6,107,2,2025-03-05,150
7,108,5,2025-03-09,700
8,109,3,2025-03-18,400
9,110,2,2025-03-28,350


In [36]:
pq("""SELECT * FROM members;""")

,member_id,name,joined_on
0,1,Anil,2023-07-01
1,2,Divya,2024-02-10
2,3,Karthik,2024-05-25
3,4,Lakshmi,2024-11-30
4,5,Nisha,2025-01-12
5,6,Sameer,2025-03-01


In [37]:
pq("""SELECT * FROM loans;  """)

,loan_id,member_id,book,loan_date,returned_on
0,1,1,Wings of Fire,2025-01-04,2025-01-18
1,2,2,Godan,2025-01-09,2025-01-30
2,3,1,Gitanjali,2025-01-22,2025-02-05
3,4,3,Malgudi Days,2025-02-03,None
4,5,2,The Guide,2025-02-11,2025-02-25
5,6,4,Train to Pakistan,2025-02-19,2025-03-10
6,7,1,Discovery of India,2025-03-02,None
7,8,5,Ignited Minds,2025-03-06,2025-03-20
8,9,2,Wise and Otherwise,2025-03-15,None
9,10,3,Swami and Friends,2025-03-27,2025-04-08


In [38]:
pq("""SELECT customer_id,                              
       COUNT(*)           AS orders,             
       SUM(amount >= 500) AS big                
FROM   shop_orders                           
GROUP  BY customer_id;   """)

,customer_id,orders,big
0,1,3,1
1,2,3,1
2,3,2,0
3,4,1,1
4,5,1,1


In [39]:
pq("""SELECT COUNT(*)           AS orders,        
       SUM(amount >= 500) AS big,                -- 4
       ROUND(100.0 * AVG(amount >= 500), 1)      
         AS pct_big                             \
FROM   shop_orders; """)

,orders,big,pct_big
0,10,4,40.0


In [40]:
pq("""
   
   SELECT COUNT(*) AS orders,
   SUM(amount>=500) as big_orders,
   ROUND(100.0*avg(amount>=500),2) as perc
   FROM shop_orders;
   """)

,orders,big_orders,perc
0,10,4,40.0


In [41]:

# On 31 March 2025, how many days had each customer been with the shop?
#customer_name ,joined_date,no of days

pq("""
   
   SELECT name,
        joined_on,
        CAST(julianday('2025-03-31')-julianday(joined_on)AS INT) AS days
        FROM shop_customers;
   """) 


,name,joined_on,days
0,Asha,2024-01-15,441
1,Ravi,2024-03-02,394
2,Meera,2024-06-20,284
3,Arjun,2024-09-10,202
4,Fatima,2025-01-05,85
5,John,2025-02-14,45


In [42]:
pq("""
   
   SELECT strftime('%Y-%m-%d-%w','2026-10-26') as n;
   
   
   """)

,n
0,2026-10-26-1


In [43]:
pq("""WITH s AS (SELECT customer_id,                   -- roll up first
             SUM(amount) AS spent                -- one total each
           FROM shop_orders GROUP BY customer_id) -- five buyers
SELECT c.name, s.spent,                          -- who, how much
       ROW_NUMBER() OVER (ORDER BY s.spent DESC) -- 1 to 5
         AS rn,                                  -- the position
       NTILE(2) OVER (ORDER BY s.spent DESC)     -- 1 or 2
         AS half                                 -- the group
FROM   s JOIN shop_customers c                   -- for the name
       ON c.customer_id = s.customer_id          -- one-to-one
ORDER  BY s.spent DESC; """)

,name,spent,rn,half
0,Ravi,1700,1,1
1,Asha,1350,2,1
2,Arjun,900,3,1
3,Fatima,700,4,2
4,Meera,650,5,2


In [44]:
#How many orders and how much sales in each month ?
#month ,order, sales 
#shop_orders

pq("""SELECT strftime('%Y-%m', order_date) AS month,   
       COUNT(*)    AS orders,                  
       SUM(amount) AS sales                      
FROM   shop_orders                           
GROUP  BY month ORDER BY month; """)


,month,orders,sales
0,2025-01,3,1950
1,2025-02,3,1750
2,2025-03,4,1600


In [45]:
# Which customers have never placed an order?

pq("""
   SELECT c.name,c.city
   FROM shop_customers AS c
   LEFT JOIN shop_orders AS o
        ON o.customer_id=c.customer_id    
    WHERE o.customer_id IS NULL;
   """)
#Which memeber have never borrowed a Book?

,name,city
0,John,Pune


In [46]:
pq("""
   SELECT m.name
   FROM members AS m
   LEFT JOIN loans AS l
        ON l.member_id=m.member_id   
    WHERE l.member_id IS NULL;
   """)

,name
0,Sameer


In [47]:
#Which customer placed at least one order on after 1st march 2025?
pq("""
   SELECT c.name 
   FROM shop_customers AS c
   WHERE EXISTS (SELECT 1 FROM shop_orders AS o
                WHERE o.customer_id=c.customer_id
                AND o.order_date>='2025-03-01');
   
   """)

,name
0,Ravi
1,Meera
2,Fatima


In [48]:
#Split the customers who bought anything into a 
# top half and a bottom half by how much they spent.
pq("""WITH s AS (SELECT customer_id,                  
             SUM(amount) AS spent                
           FROM shop_orders GROUP BY customer_id) 
SELECT c.name, s.spent,                          
       ROW_NUMBER() OVER (ORDER BY s.spent DESC) 
         AS rn,                                  
       NTILE(2) OVER (ORDER BY s.spent DESC)     
         AS half                                 
FROM   s JOIN shop_customers c                   
       ON c.customer_id = s.customer_id
ORDER  BY s.spent DESC; """)

,name,spent,rn,half
0,Ravi,1700,1,1
1,Asha,1350,2,1
2,Arjun,900,3,1
3,Fatima,700,4,2
4,Meera,650,5,2


In [49]:
pq("""SELECT order_id, order_date, amount,             -- each order
       SUM(amount) OVER (ORDER BY order_date)    -- grows down
         AS running,                             -- total so far
       SUM(amount) OVER ()                       -- no ORDER BY
         AS whole                                -- the failure
FROM   shop_orders ORDER BY order_date; """)

,order_id,order_date,amount,running,whole
0,101,2025-01-03,450,450,5300
1,103,2025-01-11,1200,1650,5300
2,102,2025-01-20,300,1950,5300
3,104,2025-02-02,250,2200,5300
4,105,2025-02-15,600,2800,5300
5,106,2025-02-21,900,3700,5300
6,107,2025-03-05,150,3850,5300
7,108,2025-03-09,700,4550,5300
8,109,2025-03-18,400,4950,5300
9,110,2025-03-28,350,5300,5300


In [50]:
pq("""SELECT loan_date, book,                         -- each loan
       COUNT(*) OVER (ORDER BY loan_date)       -- grows by one
         AS loans_so_far,                       -- 1 to 10
       SUM(returned_on IS NULL)                 -- tool 1
         OVER (ORDER BY loan_date)              -- tool 8
         AS out_so_far                          -- 0 to 3
FROM   loans ORDER BY loan_date; """)

,loan_date,book,loans_so_far,out_so_far
0,2025-01-04,Wings of Fire,1,0
1,2025-01-09,Godan,2,0
2,2025-01-22,Gitanjali,3,0
3,2025-02-03,Malgudi Days,4,1
4,2025-02-11,The Guide,5,1
5,2025-02-19,Train to Pakistan,6,1
6,2025-03-02,Discovery of India,7,2
7,2025-03-06,Ignited Minds,8,2
8,2025-03-15,Wise and Otherwise,9,3
9,2025-03-27,Swami and Friends,10,3


In [51]:
pq("""EXPLAIN QUERY PLAN                               -- how, not what
SELECT * FROM shop_orders WHERE customer_id = 2; -- one customer
  """)

,id,parent,notused,detail
0,3,0,0,SEARCH shop_orders USING INDEX ix_orders_custo...


In [52]:
pq("""EXPLAIN QUERY PLAN                               -- ask again
SELECT * FROM shop_orders WHERE customer_id = 2;""")

,id,parent,notused,detail
0,3,0,0,SEARCH shop_orders USING INDEX ix_orders_custo...


In [53]:
pq("""SELECT name, tbl_name FROM sqlite_master         -- what exists
WHERE  type = 'index'; """)

,name,tbl_name
0,ix_orders_customer,shop_orders


In [54]:
pq("""select * from shop_orders
   Where customer_id = 5 ;""")

,order_id,customer_id,order_date,amount
0,108,5,2025-03-09,700


In [55]:
pq("""EXPLAIN QUERY PLAN                              -- before
SELECT book FROM loans WHERE member_id = 1; """)

,id,parent,notused,detail
0,2,0,0,SCAN loans


In [56]:
def run_sql_file(path):
    with open(path,'r') as f:
        script=f.read()
    pcon.executescript(script)
    pcon.commit()
    print('ran',path)

In [57]:
run_sql_file(r'C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\practice\index.sql')

ran C:\Users\MD. MIDELAJ\Airtel_telecom_analytics\sql\practice\index.sql


In [60]:
pq("""SELECT name, tbl_name FROM sqlite_master         -- what exists
WHERE  type = 'index'; """)

,name,tbl_name
0,ix_orders_customer,shop_orders


In [61]:
pq("""
   SELECT * FROM shop_orders 
   WHERE customer_id=2;
   """)

,order_id,customer_id,order_date,amount
0,103,2,2025-01-11,1200
1,107,2,2025-03-05,150
2,110,2,2025-03-28,350


In [74]:
pq("""
   SELECT strftime('%m','2026-10-28') as month;
   
   """)

,month
0,10


In [67]:
pq("""
   SELECT julianday('midlaj') as id;
   """)

,id
0,None
